# LIGER default - neurips_bmmc

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'neurips_bmmc'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'batch'
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'neurips_bmmc'
assert BATCH_KEY == 'batch'
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'TRAJ', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                           neurips_bmmc
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                batch
label_key                                            cell_type
metrics      [BAMI, CASW, SCGRAPHM, TRAJ, ISOLISI, BRAS, KB...
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=90261 n_batches=12 n_labels=45
TRAJ preflight: usable_pseudotime=51407 unique_values=51251
ISOLISI preflight: min_batch_coverage=1 isolated_labels=['CD8+ T naive CD127+ CD26- CD101-', 'T prog cycling', 'cDC1']
CPCR preflight: eligible_labels=40


In [4]:
LIGER_DEFAULT_ARGS = {
    "k": 20,
    "value_lambda": 5.0,
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_default(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_DEFAULT_ARGS["k"],
        lam=LIGER_DEFAULT_ARGS["value_lambda"],
        max_iterations=LIGER_DEFAULT_ARGS["max_iters"],
        mode=LIGER_DEFAULT_ARGS["mode"],
        layer=LIGER_DEFAULT_ARGS["layer"],
        seed=seed,
    )


pd.Series(LIGER_DEFAULT_ARGS, name="default_setting")


k                   20
value_lambda       5.0
max_iters           30
mode              core
layer           counts
Name: default_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_default(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 228 genes not expressing in s2d5.
Removing 821 genes not expressing in s3d1.


Removing 47 genes not expressing in s4d1.
Removing 162 genes not expressing in s4d8.


Removing 73 genes not expressing in s4d9.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.589480712766173


optimised clustering against liger_metric_label
optimal cluster resolution: 0.4
optimal score: 0.6034051206416061
BRAS...


seed           10.000000
BAMI            0.603405
CASW            0.553858
SCGRAPHM        0.717454
TRAJ            0.568505
ISOLISI         0.958281
BRAS            0.754256
KBET            0.577649
CPCR            0.761689
bio_score       0.680300
batch_score     0.697865
joint_score     0.687326
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,TRAJ,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.563237,0.531944,0.441432,0.657413,0.912900,0.765141,0.518842,0.768837,0.621385,0.684273,0.646541
1,2,0.623708,0.558163,0.717772,0.465598,0.955709,0.765617,0.572244,0.759623,0.664190,0.699161,0.678178
2,3,0.600482,0.551085,0.689730,0.487741,0.947228,0.758731,0.574483,0.768397,0.655253,0.700537,0.673367
3,4,0.619371,0.537993,0.667970,0.530128,0.943874,0.745061,0.576934,0.749134,0.659867,0.690376,0.672071
4,5,0.584461,0.549950,0.465098,0.653255,0.941944,0.776635,0.533840,0.766028,0.638942,0.692168,0.660232
5,6,0.616518,0.555149,0.477506,0.485987,0.949259,0.769612,0.554693,0.740912,0.616884,0.688406,0.645492
6,7,0.602631,0.552233,0.494129,0.536234,0.967724,0.764996,0.513529,0.765279,0.630590,0.681268,0.650861
7,8,0.614128,0.549333,0.693016,0.511367,0.954228,0.762325,0.564421,0.752911,0.664414,0.693219,0.675936
8,9,0.600422,0.554524,0.454170,0.603958,0.949281,0.774257,0.535382,0.766766,0.632471,0.692135,0.656337
9,10,0.603405,0.553858,0.717454,0.568505,0.958281,0.754256,0.577649,0.761689,0.680300,0.697865,0.687326


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.602836,0.017150,0.563237,0.623708
CASW,0.549423,0.007759,0.531944,0.558163
SCGRAPHM,0.581828,0.116837,0.441432,0.717772
TRAJ,0.550019,0.065290,0.465598,0.657413
ISOLISI,0.948043,0.013720,0.912900,0.967724
BRAS,0.763663,0.008856,0.745061,0.776635
KBET,0.552202,0.023540,0.513529,0.577649
CPCR,0.759958,0.008912,0.740912,0.768837
bio_score,0.646430,0.020116,0.616884,0.680300
batch_score,0.691941,0.005920,0.681268,0.700537
